In [25]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/AB_NYC_2019.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()
selected_cols = [
    "id",
    "name",
    "host_id",
    "host_name",
    "neighbourhood_group",
    "neighbourhood",
    "latitude",
    "longitude",
    "room_type",
    "price",
    "minimum_nights",
    "number_of_reviews",
    "last_review",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365"
]

df_clean = df[selected_cols].copy()

df_clean.head()

Rows: 48895
Columns: 16


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [26]:
df[["id", "name", "neighbourhood_group", "neighbourhood", "room_type", "price"]].head()
df_clean.isnull().sum()

id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64

In [27]:
df["price"].isnull().sum(), df["price"].notnull().sum()
df_clean["name"] = df_clean["name"].fillna("Unknown")
df_clean["host_name"] = df_clean["host_name"].fillna("Unknown")
df_clean["reviews_per_month"] = df_clean["reviews_per_month"].fillna(0)
df_clean["last_review"] = df_clean["last_review"].fillna("No Review")

df_clean.isnull().sum()

id                                0
name                              0
host_id                           0
host_name                         0
neighbourhood_group               0
neighbourhood                     0
latitude                          0
longitude                         0
room_type                         0
price                             0
minimum_nights                    0
number_of_reviews                 0
last_review                       0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
dtype: int64

In [28]:
# Remove listings with price = 0
df_clean = df_clean[df_clean["price"] > 0]

# Remove extreme high prices using 99th percentile
upper_limit = df_clean["price"].quantile(0.99)
df_clean = df_clean[df_clean["price"] < upper_limit]

print("Upper price limit:", upper_limit)
print("Rows after removing invalid/extreme prices:", df_clean.shape[0])

Upper price limit: 799.0
Rows after removing invalid/extreme prices: 48392


In [29]:
# Feature engineering

df_clean["price_per_review"] = df_clean["price"] / (df_clean["number_of_reviews"] + 1)

df_clean["is_private_room"] = df_clean["room_type"].apply(
    lambda x: 1 if x == "Private room" else 0
)

df_clean["is_entire_home"] = df_clean["room_type"].apply(
    lambda x: 1 if x == "Entire home/apt" else 0
)

df_clean["has_reviews"] = df_clean["number_of_reviews"].apply(
    lambda x: 1 if x > 0 else 0
)

df_clean.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,price_per_review,is_private_room,is_entire_home,has_reviews
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365,14.900000,1,0,1
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355,4.891304,0,1,1
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,No Review,0.00,1,365,150.000000,1,0,0
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194,0.328413,0,1,1
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0,8.000000,0,1,1


In [30]:
df_clean.to_csv("../data/processed/airbnb_cleaned.csv", index=False)

print("Cleaned dataset saved successfully.")
print("Final shape:", df_clean.shape)

Cleaned dataset saved successfully.
Final shape: (48392, 20)
